<a href="https://colab.research.google.com/github/Adams-ke/PYTHON/blob/main/PCA_HEART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
dataset = pd.read_csv("heart.csv")
dataset.head(3)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0


In [ ]:
dataset.shape

(918, 12)

In [ ]:
dataset.iloc[0]

,0
Age,40
Sex,M
ChestPainType,ATA
RestingBP,140
Cholesterol,289
FastingBS,0
RestingECG,Normal
MaxHR,172
ExerciseAngina,N
Oldpeak,0.0


In [ ]:
df = pd.DataFrame(dataset)

In [ ]:

df['ExerciseAngina'].unique()

array(['N', 'Y'], dtype=object)

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import zscore

In [ ]:
# @title
z_scores = np.abs(zscore(dataset.select_dtypes(include=[np.number])))
threshold = 3
dataset_no_outlier = dataset[(z_scores < threshold).all(axis=1)]   # Removes rows where z_score = 3

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

label_cols   = ['Sex', 'ExerciseAngina','ST_Slope'] #label encoding
onehot_cols = ['ChestPainType', 'RestingECG'] # onehot encoding
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
# Applying labelencoder manually
# labelencoder is not native to a column transformer so It cannot be in it
le =LabelEncoder()
for col in label_cols:
  df[col] = le.fit_transform(df[col])

#defining the transformer containing the one hot encoder
Transformer = ColumnTransformer(transformers=[
    ('onehot_cols', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
    ('scaler', RobustScaler(), numeric_cols)
], remainder= 'passthrough')

transformed = Transformer.fit_transform(df)
#  Rebuild DataFrame with correct column names
ohe_feature_names = Transformer.named_transformers_['onehot_cols']\
                               .get_feature_names_out(onehot_cols).tolist()

remaining_cols = [col for col in df.columns if col not in onehot_cols]

df_transformed = pd.DataFrame(transformed, columns=ohe_feature_names + remaining_cols)

print(df_transformed.head())

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0                1.0                0.0               0.0                1.0   
1                0.0                1.0               0.0                1.0   
2                1.0                0.0               0.0                0.0   
3                0.0                0.0               0.0                1.0   
4                0.0                1.0               0.0                1.0   

   RestingECG_ST       Age  Sex  RestingBP  Cholesterol  FastingBS     MaxHR  \
0            0.0 -1.076923  0.5   0.704000          0.0   0.944444 -0.400000   
1            0.0 -0.384615  1.5  -0.458667          0.0   0.500000  0.266667   
2            1.0 -1.307692  0.0   0.640000          0.0  -1.111111 -0.400000   
3            0.0 -0.461538  0.4  -0.096000          0.0  -0.833333  0.600000   
4            0.0  0.000000  1.0  -0.298667          0.0  -0.444444 -0.400000   

   ExerciseAngina  Oldpeak  ST_Slope  

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## step 1 Define X and y
X = df_transformed.drop('HeartDisease', axis=1)
y = df_transformed['HeartDisease']

## step 2 split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 21)

## step 3 defining the models
models = {
    'Logistic Regression' :LogisticRegression(),
    'SVM':SVC(),
    'Random Forest' :RandomForestClassifier()
}

In [ ]:
# Training and predicting
results_no_pca = {}
for name, model in models.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  results_no_pca[name] = round(accuracy*100, 2)


In [ ]:
#Comparing
print("MODEL ACCURACY COMPARISON")
print("-"*34)

for name, acc in results_no_pca.items():
  print(f"{name:<21} {acc}%")

MODEL ACCURACY COMPARISON
----------------------------------
Logistic Regression   84.78%
SVM                   86.41%
Random Forest         86.41%


In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components = 0.95) #to maintain 95% of the variance
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test) # ONLY TRANSFORM NOT  FIT

print(f"Original features shape :{X_train.shape[1]}")
print(f"PCA components : {X_test_pca.shape[1]}")

Original features shape :14
PCA components : 11


In [ ]:
results_pca = {}
for name, model in models.items():
  model.fit(X_train_pca, y_train)
  y_pred = model.predict(X_test_pca)
  accuracy = accuracy_score(y_test, y_pred)
  results_pca[name] = round(accuracy*100, 2)

In [ ]:
print(f"{'Model':<25} {'Without PCA':>12} {'With PCA':>10}")
print("─" * 50)
for name in models.keys():
    print(f"{name:<25} {results_no_pca[name]:>11}% {results_pca[name]:>9}%")

Model                      Without PCA   With PCA
──────────────────────────────────────────────────
Logistic Regression             84.78%     85.87%
SVM                             86.41%     85.33%
Random Forest                   86.41%     88.59%
